# Previsão de Doenças Cardíacas com Azure Machine Learning e MLflow

Experimento completo de classificação utilizando o dataset **Heart Disease UCI**.

| Item | Detalhe |
|---|---|
| Dataset | Heart Disease UCI (1.025 registros, 13 features) |
| Objetivo | Classificar pacientes: saudável (0) ou com doença cardíaca (1) |
| Algoritmos | Random Forest e XGBoost |
| Métricas | AUC-ROC e F1-Score |
| Validação | Cross-validation estratificado (5 folds) |

## Antes de começar

Verifique se o pacote **azure-ai-ml** está instalado. Execute a célula abaixo.

> **Nota**: Se não estiver instalado, execute `pip install azure-ai-ml`.

In [ ]:
pip show azure-ai-ml

## Conectar ao Workspace

Com o SDK instalado, conecte-se ao workspace do Azure Machine Learning.

Como você está rodando em uma compute instance do Azure ML, os valores padrão são suficientes para a conexão.

In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    credential = InteractiveBrowserCredential()

In [ ]:
ml_client = MLClient.from_config(credential=credential)
print(f"Workspace: {ml_client.workspace_name}")

## Configurar o MLflow

Em uma compute instance do Azure ML, o MLflow já está instalado e integrado ao workspace.

> **Nota**: Se necessário, instale com `pip install mlflow`.

In [ ]:
pip show mlflow

## Importações

In [ ]:
import os
import tempfile
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.xgboost

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, GridSearchCV,
    cross_val_score, StratifiedKFold
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, accuracy_score,
    classification_report, confusion_matrix, RocCurveDisplay
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
print("Bibliotecas carregadas com sucesso.")

## Preparação dos Dados

O dataset está armazenado no **Azure Blob Storage** (workspaceblobstore). Realizaremos:

1. Carregamento dos dados
2. EDA — exploração, estatísticas descritivas e visualizações
3. Pré-processamento — limpeza e normalização
4. Feature Engineering

### Carregando os dados

In [ ]:
print("Carregando dados...")
try:
    path = "azureml://datastores/workspaceblobstore/paths/heart.csv"
    df = pd.read_csv(path)
    print("Fonte: Azure Blob Storage")
except Exception:
    df = pd.read_csv("../data/heart.csv")
    print("Fonte: local (fallback)")

print(f"Shape: {df.shape}")
df.head()

### EDA — Estatísticas Descritivas e Qualidade dos Dados

In [ ]:
print("=== Tipos e shape ===")
print(df.dtypes)

print("\n=== Valores nulos ===")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else "Nenhum valor nulo encontrado.")

print("\n=== Estatísticas descritivas ===")
df.describe().T

### EDA — Distribuição da Variável-Alvo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['target'].value_counts()
axes[0].bar(['Saudável (0)', 'Doente (1)'], counts.values,
            color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribuição da Variável-Alvo')
axes[0].set_ylabel('Quantidade')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Saudável', 'Doente'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporção das Classes')

plt.tight_layout()
plt.savefig("target_distribution.png")
plt.show()

### EDA — Distribuição das Features por Classe

In [ ]:
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    for target_val, color, label in [(0, '#2ecc71', 'Saudável'), (1, '#e74c3c', 'Doente')]:
        axes[i].hist(df[df['target'] == target_val][col],
                     bins=20, alpha=0.6, color=color, label=label)
    axes[i].set_title(f'Distribuição: {col}')
    axes[i].legend()

axes[-1].set_visible(False)
plt.suptitle('Distribuição das Features Numéricas por Classe', fontsize=14)
plt.tight_layout()
plt.savefig("feature_distributions.png")
plt.show()

### EDA — Boxplots e Detecção de Outliers

In [ ]:
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(16, 5))

for i, col in enumerate(numeric_cols):
    df.boxplot(column=col, by='target', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('Target')

plt.suptitle('Boxplots por Classe (0=Saudável, 1=Doente)', fontsize=13)
plt.tight_layout()
plt.savefig("boxplots.png")
plt.show()

print("=== Outliers por feature (método IQR) ===")
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f"  {col}: {outliers} outlier(s)")

### EDA — Mapa de Correlação

In [ ]:
plt.figure(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Mapa de Correlação entre Features', fontsize=14)
plt.tight_layout()
plt.savefig("correlation_heatmap.png")
plt.show()

print("\nCorrelações mais altas com 'target':")
print(corr['target'].abs().sort_values(ascending=False).drop('target'))

### Pré-processamento — Feature Engineering

Criamos quatro novas features a partir do conhecimento de domínio médico:

| Feature | Descrição |
|---|---|
| `chol_per_age` | Razão colesterol / idade (risco relativo por faixa etária) |
| `age_group` | Faixa etária (binning: ≤40, 41–55, 56–70, >70) |
| `high_bp` | Flag: pressão arterial > 140 mmHg |
| `hr_reserve` | Frequência cardíaca relativa à máxima esperada (220 - idade) |

In [ ]:
df_fe = df.copy()

df_fe['chol_per_age'] = df_fe['chol'] / df_fe['age']
df_fe['age_group']   = pd.cut(df_fe['age'], bins=[0, 40, 55, 70, 100],
                               labels=[0, 1, 2, 3]).astype(int)
df_fe['high_bp']     = (df_fe['trestbps'] > 140).astype(int)
df_fe['hr_reserve']  = df_fe['thalach'] / (220 - df_fe['age'])

print(f"Shape após feature engineering: {df_fe.shape}")
df_fe[['age', 'chol', 'thalach', 'chol_per_age', 'age_group', 'high_bp', 'hr_reserve']].head()

### Pré-processamento — Normalização e Divisão dos Dados

In [ ]:
X = df_fe.drop('target', axis=1)
y = df_fe['target']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Treino : {X_train.shape[0]} amostras")
print(f"Teste  : {X_test.shape[0]} amostras")
print(f"Treino — 0: {(y_train==0).sum()} | 1: {(y_train==1).sum()}")
print(f"Teste  — 0: {(y_test==0).sum()}  | 1: {(y_test==1).sum()}")

## Criando o Experimento MLflow

Agrupe todos os runs de treinamento dentro de um único experimento para facilitar a comparação no Azure ML Studio.

In [ ]:
experiment_name = "heart-disease-experiment"
mlflow.set_experiment(experiment_name)
print(f"Experimento: {experiment_name}")

## Treinamento e Rastreamento dos Modelos

Treinamos dois algoritmos com otimização de hiperparâmetros via **GridSearchCV**.
Cada modelo é registrado em um run MLflow com seus parâmetros, métricas e artefatos.

O cross-validation estratificado (5 folds) é usado tanto na busca de hiperparâmetros quanto na validação final.

### Random Forest — GridSearchCV + Autolog MLflow

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_params = {
    'n_estimators':    [50, 100, 200],
    'max_depth':       [3, 5, None],
    'min_samples_split': [2, 5]
}

with mlflow.start_run(run_name="Random Forest"):
    mlflow.sklearn.autolog()

    rf_grid = GridSearchCV(
        RandomForestClassifier(random_state=RANDOM_STATE),
        rf_params, cv=cv, scoring='f1', n_jobs=-1, verbose=1
    )
    rf_grid.fit(X_train, y_train)
    rf_best = rf_grid.best_estimator_

    y_pred_rf  = rf_best.predict(X_test)
    y_proba_rf = rf_best.predict_proba(X_test)[:, 1]

    rf_f1  = f1_score(y_test, y_pred_rf)
    rf_auc = roc_auc_score(y_test, y_proba_rf)
    rf_acc = accuracy_score(y_test, y_pred_rf)

    rf_cv_f1  = cross_val_score(rf_best, X_train, y_train, cv=cv, scoring='f1')
    rf_cv_auc = cross_val_score(rf_best, X_train, y_train, cv=cv, scoring='roc_auc')

    mlflow.log_metric("test_f1_score",   rf_f1)
    mlflow.log_metric("test_roc_auc",    rf_auc)
    mlflow.log_metric("test_accuracy",   rf_acc)
    mlflow.log_metric("cv_f1_mean",      rf_cv_f1.mean())
    mlflow.log_metric("cv_f1_std",       rf_cv_f1.std())
    mlflow.log_metric("cv_auc_mean",     rf_cv_auc.mean())
    mlflow.log_metric("cv_auc_std",      rf_cv_auc.std())

    # Curva ROC como artefato
    fig, ax = plt.subplots(figsize=(6, 4))
    RocCurveDisplay.from_estimator(rf_best, X_test, y_test, ax=ax, name="Random Forest")
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_title('Curva ROC — Random Forest')
    plt.savefig("ROC-Curve-RF.png")
    mlflow.log_artifact("ROC-Curve-RF.png")
    plt.show()

    print(f"Melhores hiperparâmetros RF: {rf_grid.best_params_}")
    print(f"F1 (teste): {rf_f1:.4f} | AUC (teste): {rf_auc:.4f}")
    print(f"CV F1: {rf_cv_f1.mean():.4f} ± {rf_cv_f1.std():.4f}")
    print(f"CV AUC: {rf_cv_auc.mean():.4f} ± {rf_cv_auc.std():.4f}")

Desabilitamos o autolog antes de treinar o XGBoost para demonstrar o **logging manual** — abordagem que oferece controle total sobre o que é registrado.

In [ ]:
mlflow.sklearn.autolog(disable=True)

### XGBoost — GridSearchCV + Logging Manual MLflow

In [ ]:
xgb_params = {
    'n_estimators':  [100, 200],
    'max_depth':     [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample':     [0.8, 1.0]
}

with mlflow.start_run(run_name="XGBoost"):
    xgb_grid = GridSearchCV(
        XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE),
        xgb_params, cv=cv, scoring='f1', n_jobs=-1, verbose=1
    )
    xgb_grid.fit(X_train, y_train)
    xgb_best = xgb_grid.best_estimator_

    y_pred_xgb  = xgb_best.predict(X_test)
    y_proba_xgb = xgb_best.predict_proba(X_test)[:, 1]

    xgb_f1  = f1_score(y_test, y_pred_xgb)
    xgb_auc = roc_auc_score(y_test, y_proba_xgb)
    xgb_acc = accuracy_score(y_test, y_pred_xgb)

    xgb_cv_f1  = cross_val_score(xgb_best, X_train, y_train, cv=cv, scoring='f1')
    xgb_cv_auc = cross_val_score(xgb_best, X_train, y_train, cv=cv, scoring='roc_auc')

    # Logging manual dos hiperparâmetros
    mlflow.log_params(xgb_grid.best_params_)

    # Logging manual das métricas
    mlflow.log_metric("test_f1_score",  xgb_f1)
    mlflow.log_metric("test_roc_auc",   xgb_auc)
    mlflow.log_metric("test_accuracy",  xgb_acc)
    mlflow.log_metric("cv_f1_mean",     xgb_cv_f1.mean())
    mlflow.log_metric("cv_f1_std",      xgb_cv_f1.std())
    mlflow.log_metric("cv_auc_mean",    xgb_cv_auc.mean())
    mlflow.log_metric("cv_auc_std",     xgb_cv_auc.std())

    # Curva ROC como artefato
    fig, ax = plt.subplots(figsize=(6, 4))
    RocCurveDisplay.from_estimator(xgb_best, X_test, y_test, ax=ax, name="XGBoost")
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_title('Curva ROC — XGBoost')
    plt.savefig("ROC-Curve-XGB.png")
    mlflow.log_artifact("ROC-Curve-XGB.png")
    plt.show()

    # Salva o modelo e envia como artefato
    with tempfile.TemporaryDirectory() as tmp_dir:
        model_dir = os.path.join(tmp_dir, "xgboost_model")
        mlflow.xgboost.save_model(xgb_best, model_dir)
        mlflow.log_artifacts(model_dir, artifact_path="xgboost_model")

    print(f"Melhores hiperparâmetros XGB: {xgb_grid.best_params_}")
    print(f"F1 (teste): {xgb_f1:.4f} | AUC (teste): {xgb_auc:.4f}")
    print(f"CV F1: {xgb_cv_f1.mean():.4f} ± {xgb_cv_f1.std():.4f}")
    print(f"CV AUC: {xgb_cv_auc.mean():.4f} ± {xgb_cv_auc.std():.4f}")

## Avaliação e Validação

### Comparação das Métricas no Conjunto de Teste

In [ ]:
results_test = {
    'Random Forest': {'Accuracy': accuracy_score(y_test, y_pred_rf),
                      'F1-Score': rf_f1, 'AUC-ROC': rf_auc},
    'XGBoost':       {'Accuracy': accuracy_score(y_test, y_pred_xgb),
                      'F1-Score': xgb_f1, 'AUC-ROC': xgb_auc}
}

results_cv = {
    'Random Forest': {'F1 médio (CV)': rf_cv_f1.mean(),   'F1 std': rf_cv_f1.std(),
                      'AUC médio (CV)': rf_cv_auc.mean(), 'AUC std': rf_cv_auc.std()},
    'XGBoost':       {'F1 médio (CV)': xgb_cv_f1.mean(),  'F1 std': xgb_cv_f1.std(),
                      'AUC médio (CV)': xgb_cv_auc.mean(),'AUC std': xgb_cv_auc.std()}
}

print("=== Métricas no Conjunto de Teste ===")
df_test = pd.DataFrame(results_test).T.round(4)
print(df_test)

print("\n=== Cross-Validation (5 folds) ===")
df_cv = pd.DataFrame(results_cv).T.round(4)
print(df_cv)

### Matrizes de Confusão e Relatórios de Classificação

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, (name, y_pred) in enumerate([('Random Forest', y_pred_rf),
                                     ('XGBoost', y_pred_xgb)]):
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred,
                                 target_names=['Saudável', 'Doente']))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Saudável', 'Doente'],
                yticklabels=['Saudável', 'Doente'])
    axes[i].set_title(f'Matriz de Confusão — {name}')
    axes[i].set_ylabel('Real')
    axes[i].set_xlabel('Predito')

plt.tight_layout()
plt.savefig("confusion_matrices.png")
plt.show()

### Curvas ROC — Comparação dos Modelos

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, model in [('Random Forest', rf_best), ('XGBoost', xgb_best)]:
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name)

ax.plot([0, 1], [0, 1], 'k--', label='Aleatório (AUC = 0.50)')
ax.set_title('Curvas ROC — Comparação dos Modelos')
ax.legend()
plt.tight_layout()
plt.savefig("ROC-Curves-Comparison.png")
plt.show()

### Importância das Features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, (name, model) in enumerate([('Random Forest', rf_best), ('XGBoost', xgb_best)]):
    importances = pd.Series(model.feature_importances_, index=X.columns)
    importances.sort_values().plot(kind='barh', ax=axes[i], color='steelblue')
    axes[i].set_title(f'Importância das Features — {name}')
    axes[i].set_xlabel('Importância')

plt.tight_layout()
plt.savefig("feature_importances.png")
plt.show()

## Conclusão

Revise os resultados na página **Jobs** do Azure Machine Learning Studio:

- Parâmetros: aba **Overview** → **Params**
- Métricas: aba **Metrics**
- Artefatos (curvas ROC, modelos): aba **Outputs + logs**

In [ ]:
best_model_name = max(results_test, key=lambda k: results_test[k]['AUC-ROC'])

print("=" * 60)
print("            RESUMO FINAL DO EXPERIMENTO")
print("=" * 60)
print(f"{'Modelo':<20} {'Accuracy':>10} {'F1-Score':>10} {'AUC-ROC':>10}")
print("-" * 60)
for model_name, metrics in results_test.items():
    marker = "  <-- melhor" if model_name == best_model_name else ""
    print(f"{model_name:<20} {metrics['Accuracy']:>10.4f} "
          f"{metrics['F1-Score']:>10.4f} {metrics['AUC-ROC']:>10.4f}{marker}")
print("=" * 60)
print(f"\nExperimento: {experiment_name}")
print(f"Modelo vencedor: {best_model_name}")
print("Runs disponíveis no Azure ML Studio (Jobs).")